In [ ]:
# --- repo bootstrap ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 03 — Waveform Generation

Map a smooth transport path x_c(t) to electrode voltage waveforms v(t).

```text
transport path → voltage solver → v(t) → continuity + constraints
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig
from src.ion_transport_waveform.trap_model import gaussian_electrode_basis
from src.ion_transport_waveform.transport_path import minimum_jerk_path
from src.ion_transport_waveform.waveform_solver import waveform_for_path

cfg = TrapConfig()

fig_dir = repo / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)

print("Figure dir:", fig_dir)
print("Data dir:", data_dir)


## 1. Build basis and path

This repeats the simplified segmented-electrode basis and minimum-jerk transport path used earlier.

In [ ]:
x = np.linspace(-420e-6, 420e-6, 800)
electrode_positions = np.arange(-5, 6) * cfg.electrode_pitch_m
basis = gaussian_electrode_basis(x, electrode_positions, cfg.basis_width_m)

duration = 20e-6
t = np.linspace(0, duration, 400)
path = minimum_jerk_path(t, -160e-6, 160e-6, duration=duration)

print("basis shape:", basis.shape)
print("path shape:", path.shape)
print("electrodes:", len(electrode_positions))


## 2. Generate voltage waveform

Each path sample becomes one voltage vector. The result is a waveform matrix:

```text
V[timestep, electrode]
```

In [ ]:
V = waveform_for_path(
    basis=basis,
    x_grid=x,
    path_x=path,
    voltage_limit=cfg.voltage_limit_v,
)

print("waveform shape:", V.shape)
print("max |V|:", np.max(np.abs(V)))
print("min V:", np.min(V))
print("max V:", np.max(V))


## 3. Figure: electrode voltage waveforms


In [ ]:
plt.figure(figsize=(8, 4.5))
for i in range(V.shape[1]):
    plt.plot(t * 1e6, V[:, i], alpha=0.65)

plt.xlabel("time (µs)")
plt.ylabel("voltage (V)")
plt.title("Electrode voltage waveforms")
plt.tight_layout()
plt.savefig(fig_dir / "03_waveform_voltage_profiles.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '03_waveform_voltage_profiles.png'}")


## 4. Continuity check

A waveform should not jump sharply between adjacent timesteps. Here we inspect maximum step size across electrodes.

In [ ]:
dV = np.diff(V, axis=0)
max_step_per_time = np.max(np.abs(dV), axis=1)
max_step = np.max(max_step_per_time)

dt = t[1] - t[0]
slew_rate = dV / dt
max_slew_per_time = np.max(np.abs(slew_rate), axis=1)
max_slew = np.max(max_slew_per_time)

print("max ΔV between adjacent steps:", max_step)
print("max |dV/dt|:", max_slew, "V/s")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t[1:] * 1e6, max_step_per_time)
plt.xlabel("time (µs)")
plt.ylabel("max |ΔV|")
plt.title("Waveform step size (continuity check)")
plt.tight_layout()
plt.savefig(fig_dir / "03_waveform_step_size.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '03_waveform_step_size.png'}")


## 5. Slew-rate check

Step size measures discrete continuity. Slew rate estimates voltage-change demand per second.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t[1:] * 1e6, max_slew_per_time)
plt.xlabel("time (µs)")
plt.ylabel("max |dV/dt| (V/s)")
plt.title("Waveform slew-rate demand")
plt.tight_layout()
plt.savefig(fig_dir / "03_waveform_slew_rate.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '03_waveform_slew_rate.png'}")


## 6. Save waveform data


In [ ]:
# GitHub does not preserve empty directories, so always create this before saving.
data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    data_dir / "waveform_03.npz",
    t=t,
    x_grid=x,
    electrode_positions=electrode_positions,
    path=path,
    voltages=V,
    max_step_per_time=max_step_per_time,
    max_slew_per_time=max_slew_per_time,
)

print(f"saved: {data_dir / 'waveform_03.npz'}")


## 7. Next notebook

`04_motion_simulation.ipynb` closes the loop:

```text
waveform / moving well → ion trajectory → residual excitation
```